# Movie Dataset Analysis (EDA)

This notebook analyzes `movie.csv` using Pandas, NumPy, Matplotlib, and Seaborn.

**Before running:** Keep `movie.csv` in the same folder as this notebook.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Load the dataset

In [ ]:
df = pd.read_csv("movie.csv")
df.head()

## 3. Dataset information

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nDataset information:")
df.info()
print("\nStatistical summary:")
display(df.describe(include="all"))

## 4. Check missing values

In [ ]:
df.isnull().sum()

## 5. Remove duplicate rows

In [ ]:
df = df.drop_duplicates().copy()
print("Dataset shape after removing duplicates:", df.shape)

## 6. Clean the release date and missing values

The date parser below lets Pandas infer the date format, so it works with common CSV date formats. Numeric missing values are filled with their column median; text values are filled with their mode when available.

In [ ]:
if "release_date" in df.columns:
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(include=["object", "category"]).columns
for col in categorical_cols:
    mode_values = df[col].mode()
    if not mode_values.empty:
        df[col] = df[col].fillna(mode_values[0])

print("Missing values after cleaning:")
print(df.isnull().sum())

## 7. Top 10 most popular movies

In [ ]:
top_movies = df.sort_values("popularity", ascending=False).head(10)
display(top_movies[["movie_title", "popularity"]])

## 8. Top 10 highest-rated movies (at least 100 votes)

In [ ]:
top_rated = (
    df[df["vote_count"] >= 100]
    .sort_values("vote_avg", ascending=False)
    .head(10)
)
display(top_rated[["movie_title", "vote_avg", "vote_count"]])

## 9. Movie count by language

In [ ]:
language_count = df["language"].value_counts()
display(language_count.head(10))

## 10. Top 10 popular movies — bar chart

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_movies, x="popularity", y="movie_title")
plt.title("Top 10 Most Popular Movies")
plt.xlabel("Popularity")
plt.ylabel("Movie Title")
plt.tight_layout()
plt.show()

## 11. Distribution of movie ratings

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["vote_avg"].dropna(), bins=20, kde=True)
plt.title("Distribution of Movie Ratings")
plt.xlabel("Average Rating")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

## 12. Top 10 movie languages — bar chart

In [ ]:
plt.figure(figsize=(10, 6))
language_count.head(10).plot(kind="bar")
plt.title("Top 10 Movie Languages")
plt.xlabel("Language")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. Correlation heatmap

In [ ]:
numeric_for_corr = df[["popularity", "vote_avg", "vote_count"]]
plt.figure(figsize=(8, 5))
sns.heatmap(numeric_for_corr.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Movie Dataset Correlation")
plt.tight_layout()
plt.show()

## 14. Total number of movies

In [ ]:
print("Total movies:", len(df))

## 15. Average movie rating

In [ ]:
print("Average rating:", df["vote_avg"].mean())

## 16. Most-voted movie

In [ ]:
if not df["vote_count"].dropna().empty:
    most_voted = df.loc[df["vote_count"].idxmax()]
    print("Most voted movie:", most_voted["movie_title"])
    print("Vote count:", most_voted["vote_count"])
else:
    print("Vote count data is unavailable.")

## 17. Save the cleaned dataset

In [ ]:
df.to_csv("cleaned_movie.csv", index=False)
print("Cleaned dataset saved as cleaned_movie.csv")